# Chapter 17: Maps and Business Charts

Companion notebook for *Master Data Visualization: A Hands-On Beginner's Guide with Python, Matplotlib, Seaborn & Plotly*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 17.1 Maps of US States

In [1]:
import pandas as pd
import plotly.express as px

states = pd.read_csv('states.csv')
states.head(4)

,state,population,orders,sales
0,CA,38965,1870,168300
1,TX,30503,1190,97580
2,FL,22610,927,84357
3,NY,19572,1018,83476


### `px.choropleth()`

In [2]:
fig = px.choropleth(
    states, locations='state', locationmode='USA-states',
    scope='usa', color='sales', template='simple_white',
    color_continuous_scale='Blues',
    labels={'sales': 'Sales ($)'},
    title='Sales by state, in dollars')
fig.show()

In [3]:
states['orders_per_100k'] = (
    states['orders'] / states['population'] * 100).round(1)
top = states.sort_values('orders_per_100k', ascending=False)
top[['state', 'orders_per_100k']].head(5)

,state,orders_per_100k
48,VT,8.5
12,WA,7.2
20,CO,6.6
15,MA,6.0
26,OR,5.9


In [4]:
fig = px.choropleth(
    states, locations='state', locationmode='USA-states',
    scope='usa', color='orders_per_100k',
    color_continuous_scale='Blues', template='simple_white',
    hover_name='state',
    hover_data={'state': False, 'orders': ':,',
                'population': ':,'},
    labels={'orders_per_100k': 'Orders per 100,000 people'},
    title='Vermont, Washington and Colorado order the most '
          'per person')
fig.show()

## 17.2 Treemaps

In [5]:
categories = pd.read_csv('categories.csv')
categories.groupby('group')['sales'].sum()

group
Accessories    27000
Audio          29300
Computers      88600
Name: sales, dtype: int64

### `px.treemap()`

In [6]:
fig = px.treemap(
    categories, path=['group', 'category'], values='sales',
    color='group', template='simple_white',
    color_discrete_map={'Computers': '#0b3a47',
                        'Audio': '#b26a00',
                        'Accessories': '#6fb3c2'},
    title='Computers bring in 61% of sales, led by laptops')
fig.update_traces(texttemplate='%{label}<br>$%{value:,}')
fig.data[0].textfont.color = [
    '#111' if i.startswith('Accessories') else 'white'
    for i in fig.data[0].ids]
fig.update_layout(height=500, uniformtext_minsize=9,
                  uniformtext_mode='hide')
fig.show()

## 17.3 Funnels

In [7]:
funnel = pd.read_csv('funnel.csv')
funnel['kept_pct'] = (
    funnel['people'] / funnel['people'].shift() * 100).round(0)
funnel

,stage,people,kept_pct
0,Visited the shop,48000,NaN
1,Added to cart,9600,20.0
2,Started checkout,5400,56.0
3,Paid,3240,60.0


### `px.funnel()`

In [8]:
fig = px.funnel(
    funnel, x='people', y='stage', template='simple_white',
    color_discrete_sequence=['#0b3a47'],
    labels={'people': 'People', 'stage': ''},
    title='4 in 5 visitors leave before adding to the cart')
fig.update_traces(
    texttemplate='%{x:,}<br>%{percentInitial:.0%}')
fig.show()

## 17.4 Project Timelines

In [9]:
plan = pd.read_csv('plan.csv', parse_dates=['start', 'finish'])
plan

,task,team,start,finish
0,Write page content,Marketing,2027-01-04,2027-01-22
1,Design mockups,Design,2027-01-11,2027-02-12
2,Take product photos,Marketing,2027-02-01,2027-03-05
3,Build new pages,Development,2027-02-08,2027-03-19
4,Test the checkout,Development,2027-03-15,2027-03-26
5,Launch campaign,Marketing,2027-03-29,2027-04-09


### `px.timeline()`

In [10]:
fig = px.timeline(
    plan, x_start='start', x_end='finish', y='task',
    color='team', template='simple_white',
    color_discrete_map={'Marketing': '#0b3a47',
                        'Design': '#b26a00',
                        'Development': '#6fb3c2'},
    category_orders={'task': list(plan['task'])},
    labels={'team': 'Team', 'task': ''},
    title='The checkout test ends just before the launch')
fig.show()

In [11]:
fig = px.timeline(
    plan, x_start='start', x_end='finish', y='task',
    color='team', template='simple_white',
    color_discrete_map={'Marketing': '#0b3a47',
                        'Design': '#b26a00',
                        'Development': '#6fb3c2'},
    category_orders={'task': list(plan['task'])},
    labels={'team': 'Team', 'task': ''},
    title='The checkout test ends just before the launch')
fig.add_shape(type='line', x0='2027-03-29', x1='2027-03-29',
              y0=0, y1=1, yref='paper',
              line_width=2, line_dash='dash', line_color='#000',
              opacity=1, layer='above')
fig.add_annotation(x='2027-03-29', y=1.07, yref='paper',
                   text='Launch', showarrow=False)
fig.show()

## 17.5 Waterfalls

### `go.Figure()` and `go.Waterfall()`

In [12]:
import plotly.graph_objects as go

profit = pd.read_csv('profit.csv')
type(px.bar(profit, x='item', y='amount'))

plotly.graph_objs._figure.Figure

In [13]:
steps = list(profit['item']) + ['Profit']
amounts = list(profit['amount']) + [0]
measure = ['absolute'] + ['relative'] * 5 + ['total']
total = int(profit['amount'].sum())
labels = [f'{a:,}' for a in profit['amount']] + [f'{total:,}']
total

43000

In [14]:
fig = go.Figure(go.Waterfall(
    x=steps, y=amounts, measure=measure,
    text=labels, textposition='outside',
    increasing={'marker': {'color': '#0b3a47'}},
    decreasing={'marker': {'color': '#b26a00'}},
    totals={'marker': {'color': '#6fb3c2'}},
    connector={'line': {'color': '#888', 'width': 1}}))
fig.update_layout(
    template='simple_white', showlegend=False,
    title='A month of $120,000 in sales leaves $43,000 profit',
    yaxis={'title': 'Dollars', 'tickformat': ','})
fig.show()